In [ ]:
-- 1. Table for raw MX XML messages
CREATE OR REPLACE TABLE raw_mx_messages (
    id INTEGER AUTOINCREMENT,
    message_xml VARIANT  -- Changed from XML to VARIANT
);

-- 2. Structured table for parsed values
CREATE OR REPLACE TABLE parsed_mx_messages (
    raw_id INTEGER,
    mx_type STRING,
    message_id STRING,
    creation_datetime TIMESTAMP_NTZ,
    end_to_end_id STRING,
    amount NUMBER(18,2),
    currency STRING,
    debtor_name STRING,
    creditor_name STRING
);

In [ ]:
-- Insert a sample MX message
INSERT INTO raw_mx_messages (message_xml)
SELECT PARSE_XML('<Document xmlns="urn:iso:std:iso:20022:tech:xsd:pacs.008.001.02">
    <FIToFICstmrCdtTrf>
        <GrpHdr>
            <MsgId>ABC123456</MsgId>
            <CreDtTm>2026-03-09T10:15:00</CreDtTm>
        </GrpHdr>
        <CdtTrfTxInf>
            <PmtId>
                <EndToEndId>E2EABC123456</EndToEndId>
            </PmtId>
            <IntrBkSttlmAmt Ccy="USD">1500</IntrBkSttlmAmt>
            <Dbtr><Nm>MANDAR BHALE</Nm></Dbtr>
            <Cdtr><Nm>PRADEEP DADLANI</Nm></Cdtr>
        </CdtTrfTxInf>
    </FIToFICstmrCdtTrf>
</Document>');

-- Insert a sample MX message
INSERT INTO raw_mx_messages (message_xml)
SELECT PARSE_XML('<Document xmlns="urn:iso:std:iso:20022:tech:xsd:pacs.008.001.02">
    <FIToFICstmrCdtTrf>
        <GrpHdr>
            <MsgId>ABC23456</MsgId>
            <CreDtTm>2026-03-10T10:15:00</CreDtTm>
        </GrpHdr>
        <CdtTrfTxInf>
            <PmtId>
                <EndToEndId>E2EABC23456</EndToEndId>
            </PmtId>
            <IntrBkSttlmAmt Ccy="USD">2500</IntrBkSttlmAmt>
            <Dbtr><Nm>VINOD MALLYA</Nm></Dbtr>
            <Cdtr><Nm>PRADEEP DADLANI</Nm></Cdtr>
        </CdtTrfTxInf>
    </FIToFICstmrCdtTrf>
</Document>');

-- Insert a sample MX message
INSERT INTO raw_mx_messages (message_xml)
SELECT PARSE_XML('<Document xmlns="urn:iso:std:iso:20022:tech:xsd:pacs.008.001.02">
    <FIToFICstmrCdtTrf>
        <GrpHdr>
            <MsgId>ABC567989</MsgId>
            <CreDtTm>2026-03-19T10:15:00</CreDtTm>
        </GrpHdr>
        <CdtTrfTxInf>
            <PmtId>
                <EndToEndId>E2EABC567989</EndToEndId>
            </PmtId>
            <IntrBkSttlmAmt Ccy="USD">5500</IntrBkSttlmAmt>
            <Dbtr><Nm>ASHISH CHOPRA</Nm></Dbtr>
            <Cdtr><Nm>PRADEEP DADLANI</Nm></Cdtr>
        </CdtTrfTxInf>
    </FIToFICstmrCdtTrf>
</Document>');

-- Insert a sample MX message
INSERT INTO raw_mx_messages (message_xml)
SELECT PARSE_XML('<Document xmlns="urn:iso:std:iso:20022:tech:xsd:pacs.008.001.02">
    <FIToFICstmrCdtTrf>
        <GrpHdr>
            <MsgId>ABC7896986</MsgId>
            <CreDtTm>2026-03-12T10:15:00</CreDtTm>
        </GrpHdr>
        <CdtTrfTxInf>
            <PmtId>
                <EndToEndId>E2EABC7896986</EndToEndId>
            </PmtId>
            <IntrBkSttlmAmt Ccy="USD">9800</IntrBkSttlmAmt>
            <Dbtr><Nm>MANDAR BHALE</Nm></Dbtr>
            <Cdtr><Nm>ASHISH CHOPRA</Nm></Cdtr>
        </CdtTrfTxInf>
    </FIToFICstmrCdtTrf>
</Document>');

In [ ]:
SELECT * FROM raw_mx_messages;

In [ ]:
-- Stored Procedure
CREATE OR REPLACE PROCEDURE parse_iso20022_mx()
RETURNS STRING
LANGUAGE JAVASCRIPT
EXECUTE AS OWNER
AS
$$
var stmt = snowflake.createStatement({
    sqlText: `SELECT id, TO_VARCHAR(message_xml) AS xml_str FROM raw_mx_messages`
});
var rs = stmt.execute();

var insertStmt = snowflake.createStatement({
    sqlText: `INSERT INTO parsed_mx_messages 
              (raw_id, mx_type, message_id, creation_datetime, end_to_end_id, amount, currency, debtor_name, creditor_name)
              VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)`
});

var count = 0;

while (rs.next()) {
    var rawId = rs.getColumnValue(1);
    var xmlStr = rs.getColumnValue(2);

    // Detect MX type from namespace
    var mxTypeMatch = xmlStr.match(/urn:iso:std:iso:20022:tech:xsd:([a-z]+\.\d+\.\d+\.\d+)/i);
    var mxType = mxTypeMatch ? mxTypeMatch[1] : 'UNKNOWN';

    // Remove namespace for easier parsing
    var cleanXml = xmlStr.replace(/xmlns="[^"]+"/, '');

    // Parse XML using JavaScript DOM
    var parser = new DOMParser();
    var doc = parser.parseFromString(cleanXml, "application/xml");

    function getText(tag) {
        var el = doc.getElementsByTagName(tag)[0];
        return el ? el.textContent : null;
    }
    function getAttr(tag, attr) {
        var el = doc.getElementsByTagName(tag)[0];
        return el ? el.getAttribute(attr) : null;
    }

    var messageId = getText("MsgId");
    var creationDate = getText("CreDtTm");
    var endToEndId = getText("EndToEndId");
    var amount = getText("IntrBkSttlmAmt");
    var currency = getAttr("IntrBkSttlmAmt", "Ccy");
    var debtorName = getText("Dbtr") ? doc.getElementsByTagName("Dbtr")[0].getElementsByTagName("Nm")[0].textContent : null;
    var creditorName = getText("Cdtr") ? doc.getElementsByTagName("Cdtr")[0].getElementsByTagName("Nm")[0].textContent : null;

    // Insert parsed values
    insertStmt.bind([
        rawId,
        mxType,
        messageId,
        creationDate ? creationDate : null,
        endToEndId,
        amount ? parseFloat(amount) : null,
        currency,
        debtorName,
        creditorName
    ]);
    insertStmt.execute();
    count++;
}

return `Parsed ${count} MX messages successfully.`;
$$;

In [ ]:
CREATE OR REPLACE PROCEDURE parse_iso20022_mx_snowflakenative()
RETURNS STRING
LANGUAGE SQL
AS
$$
BEGIN
    INSERT INTO parsed_mx_messages (raw_id, mx_type, message_id, creation_datetime, end_to_end_id, amount, currency, debtor_name, creditor_name)
    SELECT 
        id,
        REGEXP_SUBSTR(TO_VARCHAR(message_xml), 'urn:iso:std:iso:20022:tech:xsd:([a-z]+\\.[0-9]+\\.[0-9]+\\.[0-9]+)', 1, 1, 'i', 1),
        XMLGET(XMLGET(XMLGET(message_xml, 'FIToFICstmrCdtTrf'), 'GrpHdr'), 'MsgId'):"$"::STRING,
        TRY_TO_TIMESTAMP_NTZ(XMLGET(XMLGET(XMLGET(message_xml, 'FIToFICstmrCdtTrf'), 'GrpHdr'), 'CreDtTm'):"$"::STRING),
        XMLGET(XMLGET(XMLGET(XMLGET(message_xml, 'FIToFICstmrCdtTrf'), 'CdtTrfTxInf'), 'PmtId'), 'EndToEndId'):"$"::STRING,
        XMLGET(XMLGET(XMLGET(message_xml, 'FIToFICstmrCdtTrf'), 'CdtTrfTxInf'), 'IntrBkSttlmAmt'):"$"::NUMBER(18,2),
        XMLGET(XMLGET(XMLGET(message_xml, 'FIToFICstmrCdtTrf'), 'CdtTrfTxInf'), 'IntrBkSttlmAmt'):"@Ccy"::STRING,
        XMLGET(XMLGET(XMLGET(XMLGET(message_xml, 'FIToFICstmrCdtTrf'), 'CdtTrfTxInf'), 'Dbtr'), 'Nm'):"$"::STRING,
        XMLGET(XMLGET(XMLGET(XMLGET(message_xml, 'FIToFICstmrCdtTrf'), 'CdtTrfTxInf'), 'Cdtr'), 'Nm'):"$"::STRING
    FROM raw_mx_messages;
    
    RETURN 'Parsed ' || SQLROWCOUNT || ' MX messages successfully.';
END;
$$;

In [ ]:
CALL parse_iso20022_mx_snowflakenative();

In [ ]:
select * from parsed_mx_messages;